# Conversational Assistant and Memory

This notebook explores how conversational history is managed in LLM applications.

The objective is to understand how stateless LLM calls can be transformed into a conversational assistant and how conversation history interacts with a RAG pipeline.

In [1]:
!pip install -q \
    "google-auth==2.49.0" \
    langchain \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers


In [2]:
!pip install -q jedi

In [3]:
!pip check

No broken requirements found.


In [4]:
from google.colab import userdata

from langchain_google_genai import ChatGoogleGenerativeAI

api_key = userdata.get("GEMINI_API_KEY")

LLM_MODEL = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    api_key=api_key
)

## 1. Stateless LLM Calls

In [5]:
response_1 = llm.invoke(
    "The project codename is ZXQ-7314."
)

print(response_1.text)

Understood. I have noted the project codename **ZXQ-7314**. 

How would you like to proceed with ZXQ-7314?


In [6]:
response_2 = llm.invoke(
    "What is the project codename?"
)

print(response_2.text)

I don't have access to context or any previous conversation, so I don't know which project you are referring to. Could you please provide more details or the name of the project/company?


The second call does not contain the first one.

## 2. Conversation History

In [7]:
from langchain_core.messages import HumanMessage, AIMessage

In [43]:
history = []

In [9]:
question_1 = "The project codename is ZXQ-7314."

response_1 = llm.invoke([
    HumanMessage(content=question_1)
])

In [10]:
history.extend([
    HumanMessage(content=question_1),
    AIMessage(content=response_1.text)
])

In [11]:
for message in history:
    print(type(message).__name__)
    print(message.content)
    print()

HumanMessage
The project codename is ZXQ-7314.

AIMessage
Understood. I have logged the project codename **ZXQ-7314**. 

How would you like to proceed with ZXQ-7314 today?



In [12]:
question_2 = "What is the project codename?"

response_2 = llm.invoke(
    history + [
        HumanMessage(content=question_2)
    ]
)

print(response_2.text)

The project codename is **ZXQ-7314**.


The LLM did not remember anything. Our application remembered the conversation and sent it back to the LLM.

## 3. LangChain Chat History

In [13]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

### With history

In [14]:
chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a concise technical assistant."
    ),
    MessagesPlaceholder("history"),
    (
        "human",
        "{question}"
    )
])

In [15]:
chat_chain = (
    chat_prompt
    | llm
    | StrOutputParser()
)

In [16]:
answer = chat_chain.invoke({
    "history": history,
    "question": "What is the project codename?"
})

print(answer)

The project codename is **ZXQ-7314**.


### Independent call

In [17]:
chat_prompt_independent = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a concise technical assistant."
    ),
    (
        "human",
        "{question}"
    )
])

In [18]:
chat_chain_independent = (
    chat_prompt_independent
    | llm
    | StrOutputParser()
)

In [19]:
answer = chat_chain_independent.invoke({
    "question": "What is the project codename?"
})

print(answer)

Could you please specify which project you are referring to?


As we saw before, there is no memory of the project in the independent call

## 4. History-Aware Retrieval

### RAG Setup

In [44]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

In [45]:
texts = [
    "Amazon S3 provides object storage through buckets.",
    "Amazon EC2 provides scalable virtual computing instances.",
    "AWS Lambda runs code without managing servers.",
    "Amazon RDS provides managed relational databases.",
    "Amazon CloudFront is a content delivery network."
]

documents = [
    Document(page_content=text)
    for text in texts
]

In [46]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [47]:
vector_store = InMemoryVectorStore(embeddings)

_ = vector_store.add_documents(documents)

In [48]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

### Standalone question creation

In [49]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

contextualize_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Given the conversation history and the latest user question,
        rewrite the question so that it can be understood independently.

        Do not answer the question.
        Only return the rewritten standalone question.
        """
    ),
    MessagesPlaceholder("history"),
    (
        "human",
        "{question}"
    )
])

In [50]:
contextualize_chain = (
    contextualize_prompt
    | llm
    | StrOutputParser()
)

In [51]:
history = [
    HumanMessage(content="What is Amazon S3?"),
    AIMessage(content="Amazon S3 is an object storage service.")
]

In [52]:
follow_up = "How is it different from RDS?"

In [53]:
standalone_question = contextualize_chain.invoke({
    "history": history,
    "question": follow_up
})

print(standalone_question)

How is Amazon S3 different from Amazon RDS?


### Retrieve using the standalone question

In [54]:
retrieved_docs = retriever.invoke(standalone_question)

for doc in retrieved_docs:
    print(doc.page_content)
    print()

Amazon S3 provides object storage through buckets.

Amazon RDS provides managed relational databases.



In [55]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


In [56]:
context = format_docs(retrieved_docs)

print(context)

Amazon S3 provides object storage through buckets.

Amazon RDS provides managed relational databases.


### Create the answering prompt

In [59]:
answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Answer the user's question using only the provided context.

        If the answer cannot be found in the context, say:
        "The information is not available in the provided context."

        Context:
        {context}
        """
    ),
    MessagesPlaceholder("history"),
    (
        "human",
        "{question}"
    )
])

In [60]:
answer_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

In [61]:
answer = answer_chain.invoke({
    "context": context,
    "history": history,
    "question": follow_up
})

print(answer)

The information is not available in the provided context.


In [63]:
formatted = answer_prompt.invoke({
    "context": context,
    "history": history,
    "question": follow_up
})

print(formatted)

messages=[SystemMessage(content='\n        Answer the user\'s question using only the provided context.\n\n        If the answer cannot be found in the context, say:\n        "The information is not available in the provided context."\n\n        Context:\n        Amazon S3 provides object storage through buckets.\n\nAmazon RDS provides managed relational databases.\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is Amazon S3?', additional_kwargs={}, response_metadata={}), AIMessage(content='Amazon S3 is an object storage service.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='How is it different from RDS?', additional_kwargs={}, response_metadata={})]


## 5. Conversational RAG

In [64]:
def conversational_rag(question, history):
    standalone_question = contextualize_chain.invoke({
        "history": history,
        "question": question
    })

    retrieved_docs = retriever.invoke(standalone_question)
    context = format_docs(retrieved_docs)

    formatted = answer_prompt.invoke({
        "context": context,
        "history": history,
        "question": question
    })

    print(formatted)

    answer = answer_chain.invoke({
        "context": context,
        "history": history,
        "question": question
    })

    history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer)
    ])

    return answer

In [65]:
history = []

## 6. Multi-Turn Assistant

In [66]:
answer_1 = conversational_rag(
    "What is Amazon S3?",
    history
)

print(answer_1)

messages=[SystemMessage(content='\n        Answer the user\'s question using only the provided context.\n\n        If the answer cannot be found in the context, say:\n        "The information is not available in the provided context."\n\n        Context:\n        Amazon S3 provides object storage through buckets.\n\nAmazon CloudFront is a content delivery network.\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is Amazon S3?', additional_kwargs={}, response_metadata={})]
Based on the provided context, Amazon S3 provides object storage through buckets.


In [67]:
answer_2 = conversational_rag(
    "How is it different from RDS?",
    history
)

print(answer_2)

messages=[SystemMessage(content='\n        Answer the user\'s question using only the provided context.\n\n        If the answer cannot be found in the context, say:\n        "The information is not available in the provided context."\n\n        Context:\n        Amazon S3 provides object storage through buckets.\n\nAmazon RDS provides managed relational databases.\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is Amazon S3?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the provided context, Amazon S3 provides object storage through buckets.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='How is it different from RDS?', additional_kwargs={}, response_metadata={})]
Based on the provided context, Amazon S3 provides object storage through buckets, whereas Amazon RDS provides managed relational databases.


In [68]:
answer_3 = conversational_rag(
    "Which one should I use to store images?",
    history
)

print(answer_3)

messages=[SystemMessage(content='\n        Answer the user\'s question using only the provided context.\n\n        If the answer cannot be found in the context, say:\n        "The information is not available in the provided context."\n\n        Context:\n        Amazon S3 provides object storage through buckets.\n\nAmazon RDS provides managed relational databases.\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is Amazon S3?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the provided context, Amazon S3 provides object storage through buckets.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='How is it different from RDS?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the provided context, Amazon S3 provides object storage through buckets, whereas Amazon RDS provides managed relational databases.', additional_kwargs={}, response_metadata={

In [69]:
for message in history:
    print(type(message).__name__)
    print(message.content)
    print()

HumanMessage
What is Amazon S3?

AIMessage
Based on the provided context, Amazon S3 provides object storage through buckets.

HumanMessage
How is it different from RDS?

AIMessage
Based on the provided context, Amazon S3 provides object storage through buckets, whereas Amazon RDS provides managed relational databases.

HumanMessage
Which one should I use to store images?

AIMessage
The information is not available in the provided context.



## Key Takeaways

- LLM API calls are stateless by default; conversational memory must be managed by the application.
- Conversation history can be represented as a sequence of user and assistant messages and passed back to the model on each turn.
- Follow-up questions may be ambiguous when used directly for retrieval, so conversation history can be used to rewrite them into standalone queries.
- The retriever uses the standalone question to search the vector store and return the most relevant documents.
- Conversational RAG combines chat history, query contextualization, retrieval, and grounded generation.
- A multi-turn assistant must update its conversation history after each interaction so future questions can depend on previous turns.